# Data Integration

En esta libreta se realiza la integración de múltiples fuentes de datos a nivel municipal para construir el dataset final que será utilizado en el modelado. 

El objetivo es obtener una tabla a nivel municipio-año que combine:

- incidencia delictiva (variable objetivo)
- variables de marginación
- variables de pobreza

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
base_dir = Path().resolve().parent

data_dir = base_dir / "data"

raw_dir = data_dir / "raw"
assert raw_dir.exists() and raw_dir.is_dir()

Los datasets disponibles: 

In [3]:
for item in raw_dir.glob("*.csv"): print(item.relative_to(base_dir))

data/raw/pobreza_municipal.csv
data/raw/incidencia_delictiva_municipal.csv
data/raw/imm_2020-3.csv
data/raw/incidencia_delictiva_estatal.csv
data/raw/censo_iter_municipios_2020.csv
data/raw/pobreza_municipal_2020.csv
data/raw/indicadores_laborales_municipios.csv


Definimos las rutas a los archivos que serán utilizados:

In [4]:
path_delitos_mun = raw_dir / "incidencia_delictiva_municipal.csv"
path_imm = raw_dir / "imm_2020-3.csv"
path_pobreza = raw_dir / "pobreza_municipal.csv"

Cargamos los datasets: 

In [5]:
df_delitos = pd.read_csv(path_delitos_mun, encoding_errors='ignore')
df_marginacion = pd.read_csv(path_imm)
df_pobreza = pd.read_csv(path_pobreza)

Verificamos que los dataframes han sido cargados correctamente: 

El dataset de incidencia delictiva: 

In [6]:
df_delitos.head()

,Ao,Clave_Ent,Entidad,Cve. Municipio,Municipio,Bien jurdico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,...,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,2,...,1,1,0,1,1,0,2,1,0,1
1,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,...,0,0,0,1,0,1,0,0,0,0
2,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,...,1,1,3,2,0,1,2,0,0,0
3,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,1,...,0,1,0,0,0,0,0,0,0,0
4,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,0,...,0,0,1,0,0,0,0,0,0,0


In [7]:
df_delitos.info()

<class 'pandas.DataFrame'>
RangeIndex: 2562994 entries, 0 to 2562993
Data columns (total 21 columns):
 #   Column                 Dtype
---  ------                 -----
 0   Ao                     int64
 1   Clave_Ent              int64
 2   Entidad                str  
 3   Cve. Municipio         int64
 4   Municipio              str  
 5   Bien jurdico afectado  str  
 6   Tipo de delito         str  
 7   Subtipo de delito      str  
 8   Modalidad              str  
 9   Enero                  int64
 10  Febrero                int64
 11  Marzo                  int64
 12  Abril                  int64
 13  Mayo                   int64
 14  Junio                  int64
 15  Julio                  int64
 16  Agosto                 int64
 17  Septiembre             int64
 18  Octubre                int64
 19  Noviembre              int64
 20  Diciembre              int64
dtypes: int64(15), str(6)
memory usage: 410.6 MB


In [8]:
df_delitos.isna().sum()

Ao                       0
Clave_Ent                0
Entidad                  0
Cve. Municipio           0
Municipio                0
Bien jurdico afectado    0
Tipo de delito           0
Subtipo de delito        0
Modalidad                0
Enero                    0
Febrero                  0
Marzo                    0
Abril                    0
Mayo                     0
Junio                    0
Julio                    0
Agosto                   0
Septiembre               0
Octubre                  0
Noviembre                0
Diciembre                0
dtype: int64

* Algunas de las columnas y registros poseen errores ortográficos. Esto se debe a problemas de encoding. Durante el proceso e exploración de los datos se trato de resolver esta problemática, pero fue imposible solucionarla. En esta libreta trabajaremos considerando este error.

El dataset de marginación: 

In [9]:
df_marginacion.head()

,CVE_ENT,NOM_ENT,CVE_MUN,NOM_MUN,POB_TOT,ANALF,SBASC,OVSDE,OVSEE,OVSAE,OVPT,VHAC,PL.5000,PO2SM,IM_2020,GM_2020,IMN_2020
0,1,Aguascalientes,1001,Aguascalientes,948990,1.644738,20.367220,0.104799,0.113169,0.378610,0.591434,10.339530,7.523683,54.226594,60.318795,Muy bajo,0.944508
1,1,Aguascalientes,1002,Asientos,51536,3.526405,33.906364,2.650373,0.486448,0.858160,1.352430,22.942305,78.221049,78.565471,56.546071,Muy bajo,0.885433
2,1,Aguascalientes,1003,Calvillo,58250,4.491509,42.482450,0.365177,0.516760,0.800978,1.040411,19.219858,51.301288,79.259777,57.058251,Muy bajo,0.893453
3,1,Aguascalientes,1004,Cosío,17000,3.144867,27.696745,0.712855,0.577354,0.659833,1.030989,22.716866,65.470588,81.726369,57.114030,Muy bajo,0.894326
4,1,Aguascalientes,1005,Jesús María,129929,2.380588,26.692477,0.277034,0.354957,0.860426,1.312652,16.404575,37.164143,56.748753,59.011762,Muy bajo,0.924042


In [10]:
df_marginacion.info()

<class 'pandas.DataFrame'>
RangeIndex: 2469 entries, 0 to 2468
Data columns (total 17 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   CVE_ENT   2469 non-null   int64  
 1   NOM_ENT   2469 non-null   str    
 2   CVE_MUN   2469 non-null   int64  
 3   NOM_MUN   2469 non-null   str    
 4   POB_TOT   2469 non-null   int64  
 5   ANALF     2469 non-null   float64
 6   SBASC     2469 non-null   float64
 7   OVSDE     2469 non-null   float64
 8   OVSEE     2469 non-null   float64
 9   OVSAE     2469 non-null   float64
 10  OVPT      2469 non-null   float64
 11  VHAC      2469 non-null   float64
 12  PL.5000   2469 non-null   float64
 13  PO2SM     2469 non-null   float64
 14  IM_2020   2469 non-null   float64
 15  GM_2020   2469 non-null   str    
 16  IMN_2020  2469 non-null   float64
dtypes: float64(11), int64(3), str(3)
memory usage: 328.0 KB


El dataset de pobreza:

In [11]:
df_pobreza.head()

,clave_entidad,entidad_federativa,clave_municipio,municipio,poblacion,pobreza_porcentaje,pobreza_poblacion,pobreza_extrema_porcentaje,pobreza_extrema_poblacion,pobreza_moderada_porcentaje,...,fn_carencia_alimentacion_nutritiva_calidad_poblacion,fn_al_menos_una_carencia_porcentaje,fn_al_menos_una_carencia_poblacion,fn_al_menos_tres_carencias_porcentaje,fn_al_menos_tres_carencias_poblacion,fn_ingreso_inferior_a_lpi_porcentaje,fn_ingreso_inferior_a_lpi_poblacion,fn_ingreso_inferior_a_lpei_porcentaje,fn_ingreso_inferior_a_lpei_poblacion,entidad_federativa_etq
0,1,Aguascalientes,1001,Aguascalientes,922268,23.7,218414,2.0,18206,21.7,...,145444.0,51.6,475558.0,7.2,66231.0,33.7,310444.0,9.0,82866.0,Aguascalientes
1,1,Aguascalientes,1002,Asientos,48635,40.1,19518,4.1,2015,36.0,...,13870.0,79.7,38761.0,15.7,7650.0,46.8,22737.0,15.8,7683.0,Aguascalientes
2,1,Aguascalientes,1003,Calvillo,52377,45.8,23966,4.5,2356,41.3,...,10552.0,86.2,45164.0,14.4,7520.0,49.5,25935.0,16.0,8380.0,Aguascalientes
3,1,Aguascalientes,1004,Cosío,15942,37.0,5905,3.4,538,33.7,...,3980.0,69.3,11043.0,10.9,1736.0,47.0,7488.0,16.5,2634.0,Aguascalientes
4,1,Aguascalientes,1005,Jesús María,127962,26.3,33708,3.3,4204,23.1,...,22945.0,59.3,75894.0,11.0,14069.0,34.0,43565.0,10.9,13912.0,Aguascalientes


In [12]:
df_pobreza.info()

<class 'pandas.DataFrame'>
RangeIndex: 7382 entries, 0 to 7381
Data columns (total 73 columns):
 #   Column                                                 Non-Null Count  Dtype  
---  ------                                                 --------------  -----  
 0   clave_entidad                                          7382 non-null   int64  
 1   entidad_federativa                                     7382 non-null   str    
 2   clave_municipio                                        7382 non-null   int64  
 3   municipio                                              7382 non-null   str    
 4   poblacion                                              7382 non-null   str    
 5   pobreza_porcentaje                                     7382 non-null   str    
 6   pobreza_poblacion                                      7382 non-null   str    
 7   pobreza_extrema_porcentaje                             7382 non-null   str    
 8   pobreza_extrema_poblacion                              7382

Con lo anterior, procedemos a la preparación del dataset principal: "Incidencia Delictiva 2015-2025". 

## Preparación del dataset delitos (base)

En la libreta [`notebooks/eda_delitos.ipynb`](./eda_delitos.ipynb) observamos que cada fila reprecenta: 

> municipio + año + tipo de delito + subtipo + modalidad

Es decir, consta de delitos desagregados. Por lo tanto, hay que agregarlos para obtener el total de delitos por municipio cada año. 

Primero, obtenemos la suma de los delitos registrados por cada año para cada municipio: 

In [13]:
df_delitos['total_anual'] = df_delitos.loc[:, 'Enero':'Diciembre'].sum(axis=1)  # sum over rows
df_delitos.head()

,Ao,Clave_Ent,Entidad,Cve. Municipio,Municipio,Bien jurdico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,...,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre,total_anual
0,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,2,...,1,0,1,1,0,2,1,0,1,10
1,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,...,0,0,1,0,1,0,0,0,0,4
2,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,...,1,3,2,0,1,2,0,0,0,10
3,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,1,...,1,0,0,0,0,0,0,0,0,2
4,2015,1,Aguascalientes,1001,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,0,...,0,1,0,0,0,0,0,0,0,1


Agregamos todo por municipio-año: 

In [14]:
groupby_columns = [
    'Ao', 
    'Clave_Ent', 
    'Cve. Municipio', 
    'Entidad', 
    'Municipio'
]

df_delitos_ma = (
    df_delitos
        .groupby(groupby_columns, as_index=False)
    ['total_anual']
    .sum()
)

df_delitos_ma.head()

,Ao,Clave_Ent,Cve. Municipio,Entidad,Municipio,total_anual
0,2015,1,1001,Aguascalientes,Aguascalientes,18800
1,2015,1,1002,Aguascalientes,Asientos,247
2,2015,1,1003,Aguascalientes,Calvillo,554
3,2015,1,1004,Aguascalientes,Coso,160
4,2015,1,1005,Aguascalientes,Jess Mara,1285


El dataset `df_delitos_ma` reprecenta el total de delitos a nivel municipal para cada año. 

In [15]:
df_delitos_ma.info()

<class 'pandas.DataFrame'>
RangeIndex: 26153 entries, 0 to 26152
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Ao              26153 non-null  int64
 1   Clave_Ent       26153 non-null  int64
 2   Cve. Municipio  26153 non-null  int64
 3   Entidad         26153 non-null  str  
 4   Municipio       26153 non-null  str  
 5   total_anual     26153 non-null  int64
dtypes: int64(4), str(2)
memory usage: 1.2 MB


Verificamos elementos duplicados: 

In [16]:
df_delitos_ma.duplicated(subset=['Ao', 'Clave_Ent', 'Cve. Municipio']).sum()

np.int64(0)

In [19]:
del df_delitos

## Construcción de claves (CVEGEO)

Los datasets considerados -incidencia delictiva agregado por municipio-año, marginación y pobreza- poseen columnas con información sobre la clave de la entidad y clave del municipio. Para evitar ambigüedades con los nombres de municipios o errores debido al encoding, es conveniente construir una clave estandarizada para la integración de los datasets utilizando estas columnas. 

Definimos la columna `CVEGEO` como sigue: 

> CVEGEO = clave entidad (2 dígitos) + clave municipio (3 dígitos)

Cada dataset define de manera diferente el nombre de las columnas con información sobre la clave de entidad y clave de municipio, por lo que el procedmiento de generación de esta columna difere en cada uno. 

In [18]:
df_delitos_ma['CVEGEO'] = (
    df_delitos['Clave_Ent'].astype(str).str.zfill(2) +
    df_delitos['Cve. Municipio'].astype(str).str.zfill(3)
)

df_marginacion['CVEGEO'] = (
    df_marginacion['CVE_ENT'].astype(str).str.zfill(2) +
    df_marginacion['CVE_MUN'].astype(str).str.zfill(3)
)

df_pobreza['CVEGEO'] = (
    df_pobreza['clave_entidad'].astype(str).str.zfill(2) +
    df_pobreza['clave_municipio'].astype(str).str.zfill(3)
)

Verificamos los resultados: 

In [20]:
len(
    set(df_delitos_ma['CVEGEO'].tolist()) -
    set(df_marginacion['CVEGEO'].tolist()) -
    set(df_pobreza['CVEGEO'].tolist())
)

0

Observamos que la diferencia entre los elementos únicos en los diferentes datasets es diferente de cero. Esto confirma que la cantidad de municipios es diferente entre los datasets. 

## Selección de variables relevantes

El objetivo del proyecto es **modelar y explocar la incidencia delictiva a nivel municipal en México**, utilizando variables socioeconómicas. Mas allá de la capacidad predictiva, se busca que el modelo permita **interpretar qué factores están asociados con mayores niveles de criminalidad**. 

En este contexto, la selección de variables se realizó bajo tres criterios principales: 

* **Interpretabildiad**: variables con significado claro en términos sociales y econímicos
* **Cobertura nacional**: disponibilidad para la mayoría de los municipios
* **Reprecentatividad**: capturar distintas dimensiones socieconomicas

Las variables seleccionaadas pueden agruparse en dos grandes categorías:

1. Condiciones estructurales (margniación)

Estas variables describen las **carencias persistentes en infraestructura, educación y condiciones de vida**. 

* `IM_2020`: índice de marginación (resumen global de condiciones estructurales)
* `ANALF`: rezago educativo
* `OVSDE`: falta de drenaje
* `OVSEE`: falta de electricidad
* `VHAC`: hacinamiento
* `PO2SM`: proporción de población ocupada con bajos ingresos

Estas variables permiten capturar desigualdades estructurales que pueden influir en la incidencia delictiva a través de mecanismos como: 

* exclusión social
* limitaciones educativas
* condiciones precarias de vivienda
* baja calidad del empleo

2. Condiciones socioeconómicas (pobreza)

Estas variables reflejan la **situación económica de los hogares**, especialmente en términos de ingreso y acceso a servicios básicos.

* `fn_pobreza_porcentaje`: población en situación de pobreza
* `fn_pobreza_extrema_porcentaje`: pobreza severa
* `fn_vulnerable_ingresos_porcentaje`: vulnerabilidad económica
* `fn_carencia_servicios_basicos_vivienda_porcentaje`: carencias en servicios

Estas variables complementan los indicadores de marginación al capturar: 
* restricciones económicas
* fragilidad financiera
* condiciones de vulnerabilidad

A continuación se filtran las variables seleccionadas: 

In [21]:
# marginación
marginacion_cols = [
    'CVEGEO', 'IM_2020', 'ANALF', 'OVSDE', 'OVSEE', 'VHAC', 'PO2SM'
]

df_marginacion = df_marginacion[marginacion_cols]

# pobreza (usar columnas fn_)
pobreza_cols = [
    'CVEGEO',
    'fn_pobreza_porcentaje',
    'fn_pobreza_extrema_porcentaje',
    'fn_vulnerable_ingresos_porcentaje',
    'fn_carencia_servicios_basicos_vivienda_porcentaje'
]

df_pobreza = df_pobreza[pobreza_cols]

La combinación de ambos grupos permite construir un conjunto de variables que:

* reprecenta diferentes dimensiones del contexto socioeconómico
* evita depender de un solo indicador agregado
* facilita la interpretación de los coeficientes del modelo

En particular, se priorizaron variables en porcentaje, lo que permite comparabilidad entre municipios con distintas escalas poblacionales. 

Asimismo, se evitó incluir un número excesivo de las variables para reducir problemas de **multicolinealidad**,, dado que muchos indicadores socieconómicos tienden a estar altamente correlacionados. 

### Considerasiones sobre indicadores laborales

Se evaluó la inclusión de un dataset adicional de indicadores laborales (por ejemplo, participación económica, ocupación e informalidad). Sin embargo, este dataset presentaba cobertura limitada (\~20\% de los municipios), lo cual implicaba una pérdida significativa de información al integrarlo.

Por esta razón, se decidió excluir estos indicadores del dataset principal, priorizando la consistencia y cobertura nacional.

No obstante, la variable `PO2SM` (proporción de población ocupada con ingresos bajos), incluida en el dataset de marginación, permite capturar parcialmente esta dimensión laboral.

### Extensibilidad de las variables

La selección de variables en esta etapa corresponde a un prototipo inicial del dataset integrado. Conforme avance el proyecto, se contempla:

* evaluar la incorporación de nuevas variables (por ejemplo, laborales o financieras)
* analizar su impacto en el desempeño del modelo

De esta manera, el dataset podrá evolucionar de forma controlada.

## Integración de datasets

Para integrar los datasets, se construyó la columna `CVEGEO`. Consideraremos un `left_join` con el dataset de insidencia delictiva para perservar todos los municipios con datos de delitos y evitar perdida de observaciones. 

In [22]:
df = df_delitos_ma.merge(df_marginacion, on='CVEGEO', how='left')

df = df.merge(df_pobreza, on='CVEGEO', how='left')

Verificamos resultados: 

In [23]:
df.head()

,Ao,Clave_Ent,Cve. Municipio,Entidad,Municipio,total_anual,CVEGEO,IM_2020,ANALF,OVSDE,OVSEE,VHAC,PO2SM,fn_pobreza_porcentaje,fn_pobreza_extrema_porcentaje,fn_vulnerable_ingresos_porcentaje,fn_carencia_servicios_basicos_vivienda_porcentaje
0,2015,1,1001,Aguascalientes,Aguascalientes,18800,011001,60.318795,1.644738,0.104799,0.113169,10.33953,54.226594,23.7,2.0,10.0,1.1
1,2015,1,1001,Aguascalientes,Aguascalientes,18800,011001,60.318795,1.644738,0.104799,0.113169,10.33953,54.226594,26.1,1.6,11.5,3.0
2,2015,1,1001,Aguascalientes,Aguascalientes,18800,011001,60.318795,1.644738,0.104799,0.113169,10.33953,54.226594,30.2,2.2,9.0,3.4
3,2015,1,1002,Aguascalientes,Asientos,247,011001,60.318795,1.644738,0.104799,0.113169,10.33953,54.226594,23.7,2.0,10.0,1.1
4,2015,1,1002,Aguascalientes,Asientos,247,011001,60.318795,1.644738,0.104799,0.113169,10.33953,54.226594,26.1,1.6,11.5,3.0


In [27]:
df.tail()

,Ao,Clave_Ent,Cve. Municipio,Entidad,Municipio,total_anual,CVEGEO,IM_2020,ANALF,OVSDE,OVSEE,VHAC,PO2SM,fn_pobreza_porcentaje,fn_pobreza_extrema_porcentaje,fn_vulnerable_ingresos_porcentaje,fn_carencia_servicios_basicos_vivienda_porcentaje
78454,2025,32,32057,Zacatecas,Trancoso,283,099003,60.876953,1.104531,0.046466,0.016688,9.760754,52.84571,19.8,0.6,7.7,0.5
78455,2025,32,32057,Zacatecas,Trancoso,283,099003,60.876953,1.104531,0.046466,0.016688,9.760754,52.84571,18.2,1.3,3.1,0.9
78456,2025,32,32058,Zacatecas,Santa Mara de la Paz,28,099003,60.876953,1.104531,0.046466,0.016688,9.760754,52.84571,27.1,2.9,11.0,1.5
78457,2025,32,32058,Zacatecas,Santa Mara de la Paz,28,099003,60.876953,1.104531,0.046466,0.016688,9.760754,52.84571,19.8,0.6,7.7,0.5
78458,2025,32,32058,Zacatecas,Santa Mara de la Paz,28,099003,60.876953,1.104531,0.046466,0.016688,9.760754,52.84571,18.2,1.3,3.1,0.9


Verificamos registros duplicados: 

In [28]:
df.duplicated().sum()

np.int64(0)

Verificamos valores faltantes: 

In [29]:
df.isna().mean()

Ao                                                   0.000000
Clave_Ent                                            0.000000
Cve. Municipio                                       0.000000
Entidad                                              0.000000
Municipio                                            0.000000
total_anual                                          0.000000
CVEGEO                                               0.000000
IM_2020                                              0.000000
ANALF                                                0.000000
OVSDE                                                0.000000
OVSEE                                                0.000000
VHAC                                                 0.000000
PO2SM                                                0.000000
fn_pobreza_porcentaje                                0.006245
fn_pobreza_extrema_porcentaje                        0.006245
fn_vulnerable_ingresos_porcentaje                    0.006245
fn_caren

* Las columas con valores nulos son aquellas cuya cantidad de municipios registrados es menor a la cantidad de municipios en el dataset principal. 

El tamaño del dataset final: 

In [26]:
df.shape

(78459, 17)

In [35]:
df.columns

Index(['Ao', 'Clave_Ent', 'Cve. Municipio', 'Entidad', 'Municipio',
       'total_anual', 'CVEGEO', 'IM_2020', 'ANALF', 'OVSDE', 'OVSEE', 'VHAC',
       'PO2SM', 'fn_pobreza_porcentaje', 'fn_pobreza_extrema_porcentaje',
       'fn_vulnerable_ingresos_porcentaje',
       'fn_carencia_servicios_basicos_vivienda_porcentaje'],
      dtype='str')

Es conveniente renombrar las columnas con identicadores más explícitos que los actuales: 

In [36]:
rename_dict = {
    # identificacion
    'Ao': 'anio',
    'Clave_Ent': 'clave_entidad',
    'Cve. Municipio': 'clave_municipio',
    'Entidad': 'entidad',
    'Municipio': 'municipio',
    'total_anual': 'total_delitos',
    'CVEGEO': 'cvegeo',

    # marginacion
    'IM_2020': 'indice_marginacion',
    'ANALF': 'porcentaje_analfabetismo',
    'OVSDE': 'porcentaje_sin_drenaje',
    'OVSEE': 'porcentaje_sin_electricidad',
    'VHAC': 'porcentaje_hacinamiento',
    'PO2SM': 'porcentaje_ingresos_bajos',

    # pobreza
    'fn_pobreza_porcentaje': 'porcentaje_pobreza',
    'fn_pobreza_extrema_porcentaje': 'porcentaje_pobreza_extrema',
    'fn_vulnerable_ingresos_porcentaje': 'porcentaje_vulnerable_ingresos',
    'fn_carencia_servicios_basicos_vivienda_porcentaje': 'porcentaje_carencia_servicios_basicos'
}

In [37]:
df.rename(columns=rename_dict, inplace=True)

In [38]:
df.head()

,anio,clave_entidad,clave_municipio,entidad,municipio,total_delitos,cvegeo,indice_marginacion,porcentaje_analfabetismo,porcentaje_sin_drenaje,porcentaje_sin_electricidad,porcentaje_hacinamiento,porcentaje_ingresos_bajos,porcentaje_pobreza,porcentaje_pobreza_extrema,porcentaje_vulnerable_ingresos,porcentaje_carencia_servicios_basicos
0,2015,1,1001,Aguascalientes,Aguascalientes,18800,011001,60.318795,1.644738,0.104799,0.113169,10.33953,54.226594,23.7,2.0,10.0,1.1
1,2015,1,1001,Aguascalientes,Aguascalientes,18800,011001,60.318795,1.644738,0.104799,0.113169,10.33953,54.226594,26.1,1.6,11.5,3.0
2,2015,1,1001,Aguascalientes,Aguascalientes,18800,011001,60.318795,1.644738,0.104799,0.113169,10.33953,54.226594,30.2,2.2,9.0,3.4
3,2015,1,1002,Aguascalientes,Asientos,247,011001,60.318795,1.644738,0.104799,0.113169,10.33953,54.226594,23.7,2.0,10.0,1.1
4,2015,1,1002,Aguascalientes,Asientos,247,011001,60.318795,1.644738,0.104799,0.113169,10.33953,54.226594,26.1,1.6,11.5,3.0


Guardamos el resultado: 

In [34]:
processed_dir = data_dir / "processed"
assert processed_dir.exists() and processed_dir.is_dir()
df.to_csv(processed_dir / "dataset_model.csv", index=False)

## Conclusiones

Se construyó un dataset integrado a nivel municipio-año que combina variables delictivas y socioeconómicas. 

Este dataset servirá como base para el modelado, manteniendo consistencia geográfica mediante la clave `cvegeo` y preservando la mayor cobertura posible de municipios. 